# Prepare the default Power SCADA manual

This notebook prepares **Power SCADA Operation 2020 System Guide** for the application. It intentionally orchestrates the production package in `src/intelligent_scada/`; ingestion logic is not duplicated here. Re-running with `RESUME = True` reuses only checkpoints whose source, settings, upstream hashes and required outputs still match, while database indexing replaces the same `document_key` transactionally instead of duplicating it.

## Notebook paths

In [ ]:
import sys
from pathlib import Path

def find_project_root(start: Path) -> Path:
    candidates = (start, *start.parents)
    for candidate in candidates:
        if (candidate / 'pyproject.toml').is_file() and (candidate / 'src' / 'intelligent_scada').is_dir():
            return candidate
    raise RuntimeError('Could not locate the Intelligent SCADA Assistant project root. Open this notebook from inside the project folder.')

PROJECT_ROOT = find_project_root(Path.cwd().resolve())
SRC_DIR = PROJECT_ROOT / 'src'
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

print(f'Project root: {PROJECT_ROOT}')
print(f'Source path:  {SRC_DIR}')

## 1. Configuration

In [ ]:
from intelligent_scada.artifacts import load_embeddings
from intelligent_scada.config import Settings
from intelligent_scada.database.repository import DatabaseRepository
from intelligent_scada.embeddings.embedding_service import EmbeddingService
from intelligent_scada.logging_config import configure_logging
from intelligent_scada.pipeline import Pipeline
from intelligent_scada.rag.service import RAGService
from intelligent_scada.retrieval.retriever import Retriever

RESUME = True
FORCE = False
RUN_PROVIDER_SMOKE = False

settings = Settings.from_env(project_root=PROJECT_ROOT, env_override=True).for_default_document()
configure_logging(settings.log_level)
pipeline = Pipeline(settings)
print(f'Document: {settings.document_name}')
print(f'Document key: {settings.document_key}')
print(f'Resume: {RESUME} | Force: {FORCE}')

## 2. Verify default PDF

In [ ]:
pdf_path: Path = settings.require_default_pdf().resolve()
print(f'Expected PDF: {settings.default_pdf_display_path}')
print(f'Resolved PDF: {pdf_path}')
print(f'PDF ready: {pdf_path.is_file()}')

## 3. Analyze PDF

Image extraction/filtering is the first production ingestion stage, so it runs here immediately before PDF/TOC analysis. The retained/discarded decisions remain in the normal filter manifest for review.

In [ ]:
images = pipeline.extract_images(pdf_path, resume=RESUME, force=FORCE)
structure = pipeline.analyze_pdf(pdf_path, resume=RESUME, force=FORCE)
retained_images = [image for image in images if image['decision'] == 'keep']
print(f"Pages: {structure['total_pages']}")
print(f"TOC entries: {structure['toc_entries']}")
print(f'Retained images: {len(retained_images)}')

## 4. Extract and clean text

In [ ]:
pages = pipeline.extract_text(pdf_path, resume=RESUME, force=FORCE)
print(f'Clean page records: {len(pages)}')

## 5. Generate chunks

In [ ]:
chunks = pipeline.chunk(structure, pages, resume=RESUME, force=FORCE)
print(f'Text chunks: {len(chunks)}')

## 6. Link images

In [ ]:
links = pipeline.link_images(chunks, images, resume=RESUME, force=FORCE)
linked_image_count = sum(link['image_count'] for link in links)
print(f'Chunk/image links: {linked_image_count}')

## 7. Generate embeddings

In [ ]:
pipeline.embed(chunks, resume=RESUME, force=FORCE, show_progress=True)
embeddings = load_embeddings(settings.embeddings_npy_path)
print(f'Embeddings: {embeddings.shape[0]}')
print(f'Embedding dimensions: {embeddings.shape[1]}')

## 8. Initialize/check database

In [ ]:
if not settings.pg_conn_str:
    raise RuntimeError('PostgreSQL is not configured. Set PG_CONN_STR in .env before indexing.')

repository = DatabaseRepository(settings.pg_conn_str)
repository.apply_schema(settings.database_schema_path)
database_status = 'connected' if repository.ping() else 'unavailable'
print(f'Database: {database_status}')

## 9. Index default document

In [ ]:
index_counts = pipeline.index(pdf_path, replace=True)
print(index_counts)

## 10. Verify indexed data

In [ ]:
indexed_document = repository.get_document(settings.default_document_key)
if indexed_document is None:
    raise RuntimeError(f'Default document was not found after indexing: {settings.default_document_key}')
if indexed_document['chunk_count'] != len(chunks):
    raise RuntimeError('Indexed chunk count does not match the generated chunk artifact.')
print(f"Indexed chunks: {indexed_document['chunk_count']}")
print(f"Indexed images: {indexed_document['image_count']}")

## 11. Run a sample semantic retrieval

In [ ]:
embedding_service = EmbeddingService(
    settings.embedding_model_name,
    expected_dimension=settings.embedding_dimension,
    batch_size=settings.embedding_batch_size,
    hf_token=settings.hf_token,
    revision=settings.embedding_model_revision,
)
retriever = Retriever(settings.pg_conn_str, embedding_service, max_top_k=settings.max_top_k)
sample_results = retriever.retrieve(
    'How are alarms configured?', settings.default_document_key, top_k=3
)
for result in sample_results:
    print(f"{result['score']:.3f} | page {result['page_start']} | {result['section_title']}")

## 12. Optional provider smoke test

Explicitly set `RUN_PROVIDER_SMOKE = True` to send the question and retrieved manual excerpts to your configured answer provider. Credentials alone do not enable this external call. This request disables memory; routine memory-enabled requests also send selected history and summaries. Use only authorized source content.

In [ ]:
if RUN_PROVIDER_SMOKE and settings.answer_base_url and settings.answer_api_key:
    rag = RAGService.from_settings(settings)
    rag_result = rag.ask(
        'How are alarms configured?',
        memory_enabled=False,
        include_images=False,
        top_k=3,
    )
    print(rag_result['answer'])
    print('Citations:', [citation['label'] for citation in rag_result['citations']])
else:
    print('Skipped: provider smoke is disabled or credentials are not configured.')

## 13. Final readiness summary

In [ ]:
retrieval_ready = bool(
    sample_results
    and max(result['score'] for result in sample_results) >= settings.min_retrieval_score
)
readiness = {
    'PDF path': str(pdf_path),
    'Page count': structure['total_pages'],
    'Chunk count': len(chunks),
    'Retained image count': len(retained_images),
    'Embedding count': int(embeddings.shape[0]),
    'Embedding dimensions': int(embeddings.shape[1]),
    'Document key': settings.default_document_key,
    'Indexed database chunk count': indexed_document['chunk_count'],
    'Database status': database_status,
    'Retrieval readiness': 'ready' if retrieval_ready else 'needs review',
}
for label, value in readiness.items():
    print(f'{label}: {value}')